In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sufya6/e-commerce-customer-journey-click-to-conversion")
print("Path to dataset files:", path)

file_path = path + "/customer_journey.csv"
data = pd.read_csv(file_path)

data["Timestamp"] = pd.to_datetime(data["Timestamp"])
data["event_month"] = data["Timestamp"].dt.to_period("M")

# Cohort month = user's first seen month
data["cohort_month"] = data.groupby("UserID")["Timestamp"].transform("min").dt.to_period("M")

Active_Users_Acquisition = data.groupby("cohort_month")["UserID"].nunique()

Purchasing_Users_Count = (
    data[data["Purchased"] == True]
    .groupby("cohort_month")["UserID"]
    .nunique()
)

final = pd.DataFrame({
    "New Users": Active_Users_Acquisition,
    "Purchasers": Purchasing_Users_Count
})

final["Purchasers"] = final["Purchasers"].fillna(0).astype(int)

# Sort for plotting
final = final.sort_index()

final["Purchase Rate"] = (final["Purchasers"] / final["New Users"] * 100).round(2)

# -----------------------------
# Plot 1: Purchase rate trend
# -----------------------------
plt.figure()
plt.plot(
    final.index.astype(str),
    final["Purchase Rate"],
    marker="o"
)

plt.title("Purchase Rate by Cohort Month")
plt.xlabel("Cohort Month")
plt.ylabel("Purchase Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# -----------------------------
# Plot 2: Acquisition vs Purchasers
# -----------------------------
plt.figure()
plt.plot(
    final.index.astype(str),
    final["New Users"],
    marker="o",
    label="New Users"
)

plt.plot(
    final.index.astype(str),
    final["Purchasers"],
    marker="o",
    label="Purchasers"
)

plt.title("Acquisition vs Purchasers by Cohort Month")
plt.xlabel("Cohort Month")
plt.ylabel("User Count")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()